In [1]:
pip install gradio transformers gtts playsound speechrecognition

In [2]:
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from gtts import gTTS
import os
import speech_recognition as sr
import torch

In [ ]:
# Load model and tokenizer
model_name = "meta-llama/Llama-2-7b-chat-hf"
token = "YOUR_TOKEN-no"

# Check if CUDA is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model and tokenizer with token
model = AutoModelForCausalLM.from_pretrained(model_name, use_auth_token=token).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)

Using device: cuda


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:810: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [4]:
# Function to generate chatbot response without chat history
def chatbot_response(user_input):
    # Tokenize input and move to GPU
    inputs = tokenizer(user_input, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate response
    outputs = model.generate(**inputs)

    # Decode output
    bot_reply = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Ensure the bot doesn't repeat the user input
    if bot_reply.startswith(user_input):
        bot_reply = bot_reply[len(user_input):].strip()

    return bot_reply

In [5]:
def text_to_audio(response, voice="default"):
    # Convert text response to audio
    tts = gTTS(text=response, lang="en", slow=False)
    audio_path = "response.mp3"
    tts.save(audio_path)
    return audio_path

In [6]:
def process_input(text_input, voice_input):
    # Use speech recognition if voice input is provided
    if voice_input:
        recognizer = sr.Recognizer()
        with sr.AudioFile(voice_input) as source:
            audio = recognizer.record(source)
        try:
            # Convert speech to text
            text_input = recognizer.recognize_google(audio)
        except sr.UnknownValueError:
            text_input = "Sorry, I did not understand that."
        except sr.RequestError:
            text_input = "Sorry, there was an error with the speech service."

    # Generate AI response using chatbot_response function
    response = chatbot_response(text_input)
    audio_file = text_to_audio(response, voice_input)
    return response, audio_file

In [7]:
# Gradio interface
with gr.Blocks() as voice_assistant:
    gr.Markdown("""
    # Voice Assistant

    You can type or speak your questions, and I will respond in both text and audio.
    """)

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(label="Enter your message here:")
            voice_input = gr.Audio(type="filepath", label="Or record your voice:")
            submit_btn = gr.Button("Submit")

        with gr.Column():
            response_output = gr.Textbox(label="AI Response:")
            audio_output = gr.Audio(label="Listen to Response:")

    submit_btn.click(
        fn=process_input,
        inputs=[text_input, voice_input],
        outputs=[response_output, audio_output]
    )

In [8]:
# Launch the Gradio app
voice_assistant.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://19766a1647add399b3.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
